# Bài 10 · Làm sạch dữ liệu có cấu trúc

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu buổi học** — sau notebook này, bạn sẽ:

1. Chẩn đoán ba nhóm vấn đề: **giá trị thiếu – ngoại lai – trùng lặp**, và kiểm tra cơ chế sinh ra giá trị thiếu.
2. Xử lý theo nguyên tắc **gắn cờ trước khi xoá**; quan sát phân phối trước khi đặt ngưỡng ngoại lai.
3. Đóng gói thành **bộ quy tắc đảm bảo chất lượng (QA)**, tạo `qa_report` và phân tích độ nhạy của chỉ số.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/data/listings.csv.gz")
df = pd.read_csv(URL, usecols=[
    "id", "name", "neighbourhood_cleansed", "room_type", "price",
    "minimum_nights", "latitude", "longitude", "accommodates",
    "bedrooms", "review_scores_rating", "number_of_reviews",
])
df["price_num"] = (df["price"].str.replace("$", "", regex=False)
                              .str.replace(",", "", regex=False).astype(float))
df.shape

## 1. Giá trị thiếu — đo và kiểm tra cơ chế trước khi xử lý

In [ ]:
# Đo mức thiếu
(df.isna().mean() * 100).round(1).sort_values(ascending=False).head(5)

Tỷ lệ thiếu cho biết quy mô, chưa cho biết nguyên nhân. Trước hết cần kiểm tra mẫu hình
liên quan đến các cột khác:

| Cột | Quan sát ở mốc chụp này | Cách xử lý đề xuất |
|---|---|---|
| `review_scores_rating` (17,8%) | thiếu đúng khi `number_of_reviews == 0` | không điền 0/trung bình; chỉ tính điểm trên chỗ ở đã có đánh giá |
| `price` (4,6%) | nhóm thiếu có cơ cấu loại phòng khác nhóm đủ | loại khỏi chỉ số giá và báo cáo rõ; chưa suy đoán nguyên nhân |
| `bedrooms` (12,6%) | tỷ lệ thiếu khác rõ theo loại phòng | nếu cần điền, dùng nhóm liên quan và giữ cột cờ |

In [ ]:
# review_scores_rating: thiếu có đúng khi CHƯA có đánh giá không?
pd.crosstab(df["review_scores_rating"].isna(), df["number_of_reviews"].eq(0))

In [ ]:
# Trong mỗi loại phòng, bao nhiêu % thiếu giá? (nhẹ hơn, vẫn lệch về phòng riêng)
(df["price_num"].isna()
   .groupby(df["room_type"]).mean()
   .mul(100).round(1).sort_values(ascending=False))

In [ ]:
# ...và bao nhiêu % thiếu số phòng ngủ? (thiếu có cấu trúc: dồn ở phòng riêng)
(df["bedrooms"].isna()
   .groupby(df["room_type"]).mean()
   .mul(100).round(1).sort_values(ascending=False))

In [ ]:
# bedrooms đi theo accommodates → điền bằng trung vị NHÓM cùng sức chứa (hợp lý hơn hằng số)
muc_dien = df.groupby("accommodates")["bedrooms"].transform("median")

# Nhưng ô thiếu chủ yếu là phòng nhỏ (trung vị nhóm ~ 1) nên mean gần như không khác điền 1:
print(pd.Series({
    "mean quan sát":       df["bedrooms"].mean(),
    "mean điền hằng số 1": df["bedrooms"].fillna(1).mean(),
    "mean điền theo nhóm": df["bedrooms"].fillna(muc_dien).mean(),
}).round(2).to_string())

# Cột cờ (gắn TRƯỚC khi điền) mới là thứ cho phép loại giá trị suy diễn khỏi KPI
df["bedrooms_isna"] = df["bedrooms"].isna()
df["bedrooms"] = df["bedrooms"].fillna(muc_dien)
print("\nĐã điền", int(df["bedrooms_isna"].sum()), "dòng; còn thiếu", int(df["bedrooms"].isna().sum()))

## 2. Ngoại lai — kiểm tra miền trước, thống kê sau

In [ ]:
# Luật miền: những điều KHÔNG THỂ đúng
mien = {
    "gia_thieu": df["price_num"].isna(),
    "gia_khong_duong": df["price_num"] <= 0,
    "dem_tren_365": df["minimum_nights"] > 365,
    "toa_do_ngoai_bien": ~(df["latitude"].between(-34, -33)
                           & df["longitude"].between(-71, -70)),
}
{k: int(v.sum()) for k, v in mien.items()}

In [ ]:
# Nhìn phân phối trước khi chọn ngưỡng ngoại lai
gia = df.loc[df["price_num"] > 0, "price_num"]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].hist(gia, bins=60, color="#1E93AB")
axes[0].set_title("Phân phối trên thang giá gốc")
axes[1].hist(np.log10(gia), bins=60, color="#E8890C")
axes[1].set_title("Phân phối trên thang log10")
plt.tight_layout(); plt.show()

In [ ]:
# IQR trên phân phối lệch phải có thể gắn cờ nhiều giá trị hợp lệ
q1, q3 = gia.quantile([0.25, 0.75])
fence_iqr = q3 + 1.5 * (q3 - q1)
print(f"Ngưỡng trên IQR = {fence_iqr:,.0f} -> gắn cờ {(gia > fence_iqr).mean():.1%} số chỗ ở")

# Phân vị P99 là ngưỡng sàng lọc, không phải bằng chứng rằng giá trị là lỗi
P99_GIA_THAM_CHIEU = gia.quantile(0.99)
print(f"P99 tham chiếu = {P99_GIA_THAM_CHIEU:,.0f} -> gắn cờ {(gia > P99_GIA_THAM_CHIEU).mean():.1%}")

In [ ]:
# Kiểm tra thủ công một số dòng bị gắn cờ trước khi chọn hành động
df[df["price_num"] > P99_GIA_THAM_CHIEU].nlargest(3, "price_num")[
    ["name", "room_type", "review_scores_rating", "price_num"]]

Các giá trị trên P99 cần được kiểm tra thêm bằng nguồn gốc bản ghi, loại chỗ ở và các mốc chụp khác.
Không tự động xoá chỉ vì một giá trị nằm ở đuôi phân phối.

## 3. Trùng lặp phụ thuộc vào khoá

In [ ]:
print("Dòng trùng nguyên vẹn:", df.duplicated().sum())
print("id trùng trong mốc chụp:", df["id"].duplicated().sum())

In [ ]:
# Ghép hai mốc chụp: id lặp vì cùng chỗ ở được quan sát ở hai thời điểm
t9 = pd.read_csv("https://data.insideairbnb.com/chile/rm/santiago/"
                 "2025-09-27/visualisations/listings.csv", usecols=["id"])
t9["snapshot"], nay = "2025-09", df[["id"]].assign(snapshot="2026-06")
ca_hai = pd.concat([t9, nay])

print("id trùng khi khoá = id          :", ca_hai["id"].duplicated().sum())
print("trùng khi khoá = (id, snapshot) :", ca_hai.duplicated().sum())

12.429 `id` xuất hiện ở cả hai mốc chụp: cùng một chỗ ở tại hai thời điểm. Đây là
dữ liệu bảng theo thời gian (panel data), với khoá của bảng gộp là `(id, snapshot)`.

## 4. Đóng gói: bộ quy tắc QA + báo cáo tác động

In [ ]:
def qa_rules(d, *, p99_gia=P99_GIA_THAM_CHIEU):
    """Mỗi quy tắc là một mặt nạ; hàm chỉ phát hiện, không sửa dữ liệu."""
    gia = d["price_num"]
    rules = {
        "gia_thieu":     gia.isna(),
        "gia_tren_p99_ref": gia > p99_gia,
        "dem_tren_365":     d["minimum_nights"] > 365,
        "toa_do_ngoai_bien": ~(d["latitude"].between(-34, -33)
                           & d["longitude"].between(-71, -70)),
        "id_trung":      d["id"].duplicated(keep=False),
        "bedrooms_dien": d["bedrooms_isna"],
    }
    return rules

report = pd.DataFrame([
    {"quy_tac": ten, "so_dong": int(m.sum()), "ty_le_%": round(m.mean() * 100, 2)}
    for ten, m in qa_rules(df).items()
])
report

In [ ]:
# Gắn cờ tổng hợp và phân tích độ nhạy của chỉ số giá
masks = qa_rules(df)
df["flag_gia"] = masks["gia_thieu"] | masks["gia_tren_p99_ref"]

truoc = df["price_num"].agg(["mean", "median"])
tam_loai_co = df.loc[~df["flag_gia"], "price_num"].agg(["mean", "median"])
pd.DataFrame({"mọi dòng có giá": truoc, "tạm loại dòng bị cờ": tam_loai_co}).round(0)

Cờ giá gồm **846 dòng thiếu** và **177 dòng trên P99 tham chiếu**. Các dòng thiếu vốn
không tham gia phép tính; khi tạm loại thêm 177 giá cao, trung bình thay đổi rõ còn trung vị
chỉ thay đổi nhẹ. Đây là phân tích độ nhạy, không phải quyết định xoá tự động.

## 5. Bài tập tại lớp

### Bài 1 — Thêm 2 quy tắc mới

Bổ sung vào `qa_rules`: (a) `diem_ngoai_thang` — `review_scores_rating` ngoài [0, 5];
(b) `ten_trong` — cột `name` rỗng hoặc chỉ toàn khoảng trắng. Chạy lại `report`.

In [ ]:
# TODO Bài 1:
def qa_rules_v2(d):
    r = qa_rules(d)
    r["diem_ngoai_thang"] = ~d["review_scores_rating"].between(0, 5) & d["review_scores_rating"].notna()
    r["ten_trong"] = d["name"].isna() | (d["name"].str.strip().str.len() == 0)
    return r

pd.DataFrame([{"quy_tac": t, "so_dong": int(m.sum())} for t, m in qa_rules_v2(df).items()])

### Bài 2 — Ngoại lai trong từng quận

Quy tắc `gia_tren_p99_ref` dùng ngưỡng **toàn thành phố**, nên chưa phản ánh mặt bằng giá
khác nhau giữa các quận. Viết quy tắc `gia_20x_quan`: giá gấp hơn 20 lần **trung vị quận**.
Bao nhiêu dòng bị cờ? So với `gia_tren_p99_ref`, hai quy tắc bắt trùng nhau
bao nhiêu dòng?

In [ ]:
# TODO Bài 2:
he_so = df["price_num"] / df.groupby("neighbourhood_cleansed")["price_num"].transform("median")
gia_20x = he_so > 20
print("gia_20x_quan bắt:", int(gia_20x.sum()))
print("giao với gia_tren_p99_ref:", int((gia_20x & masks["gia_tren_p99_ref"]).sum()))

### Bài 3 — Bộ QA chạy trên mốc chụp khác

Chạy `qa_rules` trên mốc 09/2025 (bản `visualisations` có **cấu trúc cột khác**: giá là số,
không có `bedrooms`). Viết phiên bản phòng thủ: chỉ chạy quy tắc khi cột cần thiết tồn tại,
và dùng lại ngưỡng giá đã hiệu chỉnh trên mốc tham chiếu thay vì tính lại P99.

In [ ]:
# TODO Bài 3 (scaffold):
t9_full = pd.read_csv("https://data.insideairbnb.com/chile/rm/santiago/"
                      "2025-09-27/visualisations/listings.csv")
t9_full["price_num"] = t9_full["price"]      # bản này giá là số sẵn

def qa_rules_defensive(d, *, p99_gia=P99_GIA_THAM_CHIEU):
    rules = {}
    if "price_num" in d:
        rules["gia_thieu"] = d["price_num"].isna()
        rules["gia_tren_p99_ref"] = d["price_num"] > p99_gia
    if "minimum_nights" in d:
        rules["dem_tren_365"] = d["minimum_nights"] > 365
    if "id" in d:
        rules["id_trung"] = d["id"].duplicated(keep=False)
    return rules

pd.DataFrame([{"quy_tac": t, "so_dong": int(m.sum())}
              for t, m in qa_rules_defensive(t9_full).items()])

## 6. Bài tập về nhà — Bộ QA cho thành phố của nhóm

Lấy thành phố nhóm bạn (dự kiến) nhận cho bài tập lớn:

1. Tải `listings.csv.gz` ở mốc chụp mới nhất; chẩn đoán ba nhóm vấn đề như notebook này.
2. Đề xuất **≥8 quy tắc QA** theo bảng 4 phần (tên – điều kiện – lý do – hành động) —
   trong đó ít nhất 2 quy tắc *đặc thù cho thành phố đó* (gợi ý: biên toạ độ, tiền tệ,
   mùa vụ, quy định địa phương về giấy phép…).
3. Sinh `qa_report.csv` và bảng phân tích độ nhạy của chỉ số giá.
4. Ghi lại quy tắc còn gây tranh luận trong nhóm và lý do chọn cách xử lý hiện tại.

In [ ]:
RUN_CHALLENGE = False

if RUN_CHALLENGE:
    CITY_URL = "https://data.insideairbnb.com/.../listings.csv.gz"   # đổi theo nhóm
    ...

---

## Tóm tắt buổi học

| Nội dung chính | Vì sao quan trọng |
|---|---|
| Gắn cờ trước khi xoá; quy tắc đủ bốn phần | Giữ dấu vết và giải thích được quyết định |
| Kiểm tra cơ chế thiếu; giá trị suy diễn phải có cờ | Phân biệt dữ liệu quan sát với dữ liệu được điền |
| Quan sát phân phối rồi mới đặt ngưỡng ngoại lai | Ngưỡng sàng lọc không bị hiểu nhầm thành bằng chứng lỗi |
| Trùng lặp phụ thuộc khoá; dữ liệu bảng dùng `(id, snapshot)` | Giữ đúng nhiều lần quan sát của một chỗ ở |
| `qa_report` và phân tích độ nhạy | Đo tác động của từng lựa chọn xử lý |

**Buổi sau:** xử lý dữ liệu phi cấu trúc bằng mô hình ngôn ngữ lớn (LLM). Hướng dẫn tạo
API key nằm ở đầu notebook Bài 11.